# LecGap Phase 3 — MiniLM cross-encoder fine-tune (GPU)Fine-tunes a cross-encoder transformer over **LectureBank 1.0** prerequisite pairs (backbone **`sentence-transformers/all-MiniLM-L6-v2`**) and evaluates it with the same nested 5-fold CV used locally.**Input (Kaggle dataset):** two CSVs are expected at`/kaggle/input/datasets/ayushdevadiga/lecturebank/`    - `prerequisite_annotation.csv` — `(Source_Topic_ID, Target_Topic_ID, If_prerequisite)`    - `208topics.csv` — `(id, Topic, Topic_Link)`If your dataset lives at a different path, update `INPUT_DIR` in the training cell below.**Output:** training writes to `/kaggle/working/staging/` (model + `metrics.json`,with `UNEXPECTED`/`MISSING` load-report noise suppressed). The **last cell** — whichyou run **manually after verifying the CV F1 beats the frozen baseline (0.569)** —copies the model into the shipped `/kaggle/working/model/` folder. Download that`model/` folder and load it on CPU for inference in the LecGap pipeline.Set **Accelerator = GPU T4** and **Internet = On** (Internet is only needed todownload the pretrained checkpoint; the notebook code itself is embedded).

In [ ]:
import torchprint('GPU available:', torch.cuda.is_available())print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
!pip install -q transformers sentence-transformers datasets scikit-learn

### 1. Core module — training/export helpersThis cell writes `experiments/fine_tune.py` + its `backend.pipeline.model_ids`and `backend.config` dependencies to disk (so the subprocess can import them)and loads the module.

In [ ]:
import osos.makedirs('/kaggle/working/experiments', exist_ok=True)os.makedirs('/kaggle/working/backend/pipeline', exist_ok=True)open('/kaggle/working/experiments/__init__.py', 'w').close()fine_tune_src = (r'''"""Phase 3 — fine-tuning the pretrained encoder on LectureBank pairs.Builds on the frozen-encoder baseline (classify_prerequisites.py) by actuallytraining the transformer over labeled (A, B) pairs, per plan/EVALUATION.md"fine-tune a pretrained embedding model (not train from scratch)".Design:  * Cross-encoder style: loads the MiniLM checkpoint and adds a binary    sequence-classification head, then fine-tunes ALL weights over the pairs.    This is trained with a plain PyTorch loop (no trainer/datasets coupling) so    the exact same code runs locally on CPU and on Kaggle GPU.  * Paired texts are fed as "[A] [SEP] [B]" so the model attends jointly, i.e.    it can answer "is A a prerequisite of B?" — the ordering matters and the    model sees it.  * Deterministic seed for reproducible CV."""from typing import List, Optional, Sequence, Tupleimport osfrom pathlib import Pathimport numpy as npfrom backend.pipeline.model_ids import EMBEDDING_MODEL, load_kwargs# Keep the transformers LOAD REPORT (UNEXPECTED / MISSING keys emitted when loading# a sentence-encoder checkpoint into a sequence-classification head) out of the log.# The MISSING classifier weights are intentionally freshly-initialized head params# and the UNEXPECTED position_ids are a benign task-shape mismatch — both expected.import logging as _loggingimport transformers as _transformers_transformers.logging.set_verbosity_error()_logging.getLogger("transformers").setLevel(_logging.ERROR)# L10: the fine-tune base mirrors the pinned runtime encoder (env-overridable)._DEF_BASE = EMBEDDING_MODELdef build_train_triples(    pairs: Sequence[Tuple[str, str]],    labels: Sequence[int],    *,    max_neg_ratio: int = 8,    random_state: Optional[int] = None,) -> List[Tuple[str, str, int]]:    """Undersample negatives to max_neg_ratio:1 vs positives.    Returns [(text_a, text_b, label), ...] ready for training.    """    pos = [(p[0], p[1], 1) for p, l in zip(pairs, labels) if l == 1]    neg = [(p[0], p[1], 0) for p, l in zip(pairs, labels) if l == 0]    import random as _random    rng = _random.Random(random_state)    neg = rng.sample(neg, min(len(neg), len(pos) * max_neg_ratio))    all_ = pos + neg    rng.shuffle(all_)    return all_def _pair_text(a: str, b: str) -> str:    """Format a pair for the cross-encoder. Order is meaningful (A precedes B)."""    return f"{a} [SEP] {b}"def _build_model(base_model: str, device: str):    import torch    from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer    tokenizer = AutoTokenizer.from_pretrained(base_model, **load_kwargs(base_model))    # Convert the ST/MPNet checkpoint into a binary sequence-classification    # head, keeping its pretrained weights (they are loaded as the base).    config = AutoConfig.from_pretrained(base_model, num_labels=1)    model = AutoModelForSequenceClassification.from_pretrained(        base_model, config=config, ignore_mismatched_sizes=True, **load_kwargs(base_model)    )    model.to(device)    return model, tokenizerdef fine_tune_cross_encoder(    train_triples: Sequence[Tuple[str, str, int]],    *,    base_model: str = _DEF_BASE,    val_triples: Optional[Sequence[Tuple[str, str, int]]] = None,    epochs: int = 3,    batch_size: int = 32,    lr: float = 2e-5,    weight_decay: float = 0.0,    grad_clip: Optional[float] = None,    seed: int = 42,    device: str = None,):    """Fine-tune a (cross-encoder) transformer on (a, b, label) triples.    Returns (model, tokenizer) with the trained weights. The full network    (transformer body + classification head) is trained — this is the    "fine-tune, don't train from scratch" step the plan calls for.    ``weight_decay`` (L2 on non-bias/norm params, per common practice) and    ``grad_clip`` (max gradient norm) help fight the overfitting/instability    seen on the small LectureBank positive pool.    """    import torch    from torch.utils.data import DataLoader, TensorDataset    from torch.optim import AdamW    from torch.nn import BCEWithLogitsLoss    if device is None:        device = "cuda" if torch.cuda.is_available() else "cpu"    torch.manual_seed(seed)    np.random.seed(seed)    model, tokenizer = _build_model(base_model, device)    # Encode all texts once.    texts = [ _pair_text(a, b) for a, b, _ in train_triples ]    enc = tokenizer(        texts,        padding=True,        truncation=True,        max_length=128,        return_tensors="pt",    )    labels = torch.tensor([float(l) for _, _, l in train_triples], dtype=torch.float)    dataset = TensorDataset(enc["input_ids"], enc["attention_mask"], labels)    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)    # Apply weight decay only to 2D (weight) params, not biases/norms — standard.    decay = [p for n, p in model.named_parameters() if p.dim() >= 2]    no_decay = [p for n, p in model.named_parameters() if p.dim() < 2]    optimizer = AdamW(        [            {"params": decay, "weight_decay": weight_decay},            {"params": no_decay, "weight_decay": 0.0},        ],        lr=lr,    )    loss_fn = BCEWithLogitsLoss()    model.train()    for epoch in range(epochs):        total = 0.0        for step, (ids, mask, lbl) in enumerate(loader):            ids, mask, lbl = ids.to(device), mask.to(device), lbl.to(device)            optimizer.zero_grad()            logits = model(input_ids=ids, attention_mask=mask, labels=None).logits            loss = loss_fn(logits.squeeze(-1), lbl)            loss.backward()            if grad_clip is not None:                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)            optimizer.step()            total += loss.item()        avg = total / max(1, len(loader))        # (val logging intentionally omitted — eval is done by evaluate_classifier.)    return model, tokenizerdef export_model(model, tokenizer, output_dir: str) -> str:    """Save the fine-tuned transformer + tokenizer; return the path."""    model.save_pretrained(output_dir)    tokenizer.save_pretrained(output_dir)    return output_dirdef _validate_model_dir(model_dir: str) -> str:    """Validate a model directory before handing it to transformers (#21).    Rejects NUL bytes and ``..`` traversal, requires an existing directory, and    — when ``LECGAP_MODEL_ROOTS`` is set (os.pathsep-separated) — requires the    resolved directory to live under one of those trusted roots.    """    raw = str(model_dir).strip()    if not raw or "\x00" in raw:        raise ValueError("model_dir must be a non-empty path")    p = Path(raw)    if ".." in p.parts:        raise ValueError(f"model_dir must not contain '..': {model_dir!r}")    resolved = p.resolve()    if not resolved.is_dir():        raise ValueError(f"model_dir is not a directory: {model_dir!r}")    roots = os.getenv("LECGAP_MODEL_ROOTS", "").strip()    if roots:        allowed = [Path(r).resolve() for r in roots.split(os.pathsep) if r.strip()]        if not any(resolved == root or root in resolved.parents for root in allowed):            raise ValueError(f"model_dir outside LECGAP_MODEL_ROOTS: {model_dir!r}")    return str(resolved)def load_model(model_dir: str, device: str = None):    import torch    from transformers import AutoModelForSequenceClassification, AutoTokenizer    model_dir = _validate_model_dir(model_dir)    if device is None:        device = "cuda" if torch.cuda.is_available() else "cpu"    model = AutoModelForSequenceClassification.from_pretrained(model_dir)    model.to(device)    model.eval()    tokenizer = AutoTokenizer.from_pretrained(model_dir)    return model, tokenizerdef predict_pairs(model, tokenizer, pairs: Sequence[Tuple[str, str]]) -> np.ndarray:    """Return a per-pair logit; higher = A is more likely a prereq of B."""    import torch    texts = [_pair_text(a, b) for a, b in pairs]    enc = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")    device = next(model.parameters()).device    enc = {k: v.to(device) for k, v in enc.items()}    with torch.no_grad():        logits = model(**enc).logits    return logits.squeeze(-1).detach().cpu().numpy()''')with open('/kaggle/working/experiments/fine_tune.py', 'w') as _f:    _f.write(fine_tune_src)model_ids_src = (r'''"""Model-ID pinning (P11/L10).The hub identifiers below are the single source of truth for which encoder thepipeline loads off HuggingFace, so a deployment can fix one model name *and*one commit revision instead of silently tracking a moving hub artifact. Bothare env-overridable:  LECGAP_EMBEDDING_MODEL    e.g. "sentence-transformers/all-mpnet-base-v2"  LECGAP_EMBEDDING_REVISION e.g. "<git commit sha>" — empty by default; set it                            to freeze the exact hub snapshot (reproducible,                            CVE-stable pinning rather than latest-at-runtime).Every runtime ``SentenceTransformer(...)`` load site routes through these twovalues (see ``load_kwargs``), so the running pipeline never silently switchesto a moved hub artifact."""from backend.config import EMBEDDING_MODEL, EMBEDDING_REVISIONdef load_kwargs(model: str = None) -> dict:    """Keyword args to forward to a ``SentenceTransformer(...)`` load site.    The pinned hub revision is applied only when the model being loaded IS the    pinned model; a caller that deliberately overrides the model id keeps    control of its own revision. Empty by default so the load behaves exactly    as before unless an operator opts into revision pinning.    """    if (model or EMBEDDING_MODEL) == EMBEDDING_MODEL and EMBEDDING_REVISION:        return {"revision": EMBEDDING_REVISION}    return {}''')with open('/kaggle/working/backend/pipeline/model_ids.py', 'w') as _f:    _f.write(model_ids_src)config_src = (r'''"""Central configuration — the single source of truth for the runtime's env surface.Every environment variable the serving system reads flows through this module,so exactly one file has to answer "what does the app read from the environment,and with what defaults/typing?" Keep ``.env.example`` in sync when a key isadded or a default changes.Two ways in:  * Import-time values (the models/route constants frozen at process start)    are exported as module constants below — same read timing as before.  * Values that should reflect a runtime change to the environment use the    lazy getter functions inside the code that calls them (``groq_api_key()``    etc.). Use the typed getters (``get_int`` / ``get_float`` / ``get_bool``)    so parsing and bounds-checking live in exactly one file."""import osfrom pathlib import PathREPO_ROOT = Path(__file__).resolve().parents[1]# ------------------------------------------------------------- typed gettersdef get_str(name: str, default: str = "") -> str:    return (os.getenv(name) or "").strip() if name in os.environ else defaultdef get_int(name: str, default: int) -> int:    raw = os.getenv(name, "").strip()    if not raw:        return default    try:        return int(raw)    except ValueError:        return defaultdef get_float(name: str, default: float) -> float:    raw = os.getenv(name, "").strip()    if not raw:        return default    try:        return float(raw)    except ValueError:        return defaultdef get_bool(name: str) -> bool:    return os.getenv(name, "").strip().lower() in {"1", "true", "yes"}# ------------------------------------------------------- lazy (call-time) keysdef api_key() -> str:    """LECGAP_API_KEY — bearer/YAML-style auth secret; empty means 'open'."""    return get_str("LECGAP_API_KEY")def groq_api_key() -> str:    return get_str("GROQ_API_KEY")def llm_reasoning_enabled() -> bool:    return get_bool("LECGAP_LLM_REASONING")def snap_silence_enabled() -> bool:    return get_bool("LECGAP_SNAP_SILENCE")def clip_streamcopy_enabled() -> bool:    return get_bool("LECGAP_CLIP_STREAMCOPY")def clip_reencode_threshold_s() -> float:    return get_float("LECGAP_CLIP_REENCODE_THRESHOLD_S", 120.0)def clip_workers() -> int | None:    """LECGAP_CLIP_WORKERS, validated; None means 'auto' (use logical cores)."""    raw = get_str("LECGAP_CLIP_WORKERS")    if not raw:        return None    try:        return max(1, int(raw))    except ValueError:        return Nonedef whisper_backend_override() -> str:    """WHISPER_BACKEND as explicitly set (empty when unconfigured)."""    return get_str("WHISPER_BACKEND")# --------------------------------------------------- import-time (frozen) keys# Database (backend/models/db.py). The pragma listener keys on the scheme,# so keep the sqlite:// default shape unchanged.DATABASE_URL = get_str(    "LECGAP_DATABASE_URL", f"sqlite:///{REPO_ROOT / 'data' / 'lecgap.db'}")# Upload cap (MiB) — 0 disables the cap for local use (backend/api/routes/lectures.py).MAX_UPLOAD_MB = get_int("LECGAP_MAX_UPLOAD_MB", 2048)# Concurrent heavy pipeline jobs — >= 1 (backend/api/jobs/common.py).MAX_PIPELINE_JOBS = max(1, get_int("LECGAP_MAX_PIPELINE_JOBS", 3))# LLM gateway (backend/pipeline/llm.py).GROQ_MODEL = get_str("LECGAP_GROQ_MODEL", "openai/gpt-oss-20b")OLLAMA_MODEL = get_str("LECGAP_OLLAMA_MODEL", "llama3.2")OLLAMA_BASE_URL = get_str("LECGAP_OLLAMA_URL", "http://127.0.0.1:11434")LLM_MAX_RETRIES = get_int("LECGAP_LLM_RETRIES", 2)LLM_SLEEP_CAP_S = get_float("LECGAP_LLM_SLEEP_CAP_S", 120.0)LLM_CACHE_TTL_S = get_float("LECGAP_LLM_CACHE_TTL_S", 30 * 24 * 3600)# Hub encoder pinning (backend/pipeline/model_ids.py).EMBEDDING_MODEL = get_str(    "LECGAP_EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")EMBEDDING_REVISION = get_str("LECGAP_EMBEDDING_REVISION") or None# Lecture-structure pass (backend/pipeline/passages.py).STRUCTURE_WINDOW_CHARS = get_int("LECGAP_STRUCTURE_WINDOW_CHARS", 8000)STRUCTURE_OVERLAP_FRAC = get_float("LECGAP_STRUCTURE_OVERLAP_FRAC", 0.25)STRUCTURE_MAX_TOKENS = get_int("LECGAP_STRUCTURE_MAX_TOKENS", 3000)# Transcription (backend/pipeline/transcribe.py).WHISPER_BACKEND = get_str("WHISPER_BACKEND", "local")WHISPER_MODEL = get_str("WHISPER_MODEL", "base")GROQ_WHISPER_MODEL = get_str("GROQ_WHISPER_MODEL", "whisper-large-v3-turbo")GROQ_WHISPER_UPLOAD_LIMIT = get_int("GROQ_WHISPER_UPLOAD_LIMIT", 24 * 1024 * 1024)GROQ_WHISPER_MAX_CHUNK_S = get_int("GROQ_WHISPER_MAX_CHUNK_S", 300)WHISPER_MAX_RETRIES = get_int("LECGAP_WHISPER_RETRIES", 2)WHISPER_500_BACKOFF_S = get_float("LECGAP_WHISPER_500_BACKOFF", 5.0)''')with open('/kaggle/working/backend/config.py', 'w') as _f:    _f.write(config_src)print('wrote experiments/fine_tune.py + backend/pipeline/model_ids.py + backend/config.py')

### 2. Training + evaluation scriptThis cell writes `scripts/kaggle_fine_tune.py` (nested 5-fold CV plus final model export)to `/kaggle/working/`.

In [ ]:
kaggle_src = r'''"""Phase 3 — fine-tune the encoder and evaluate 5-fold CV, GPU-or-CPU agnostic.This is the training+benchmark counterpart to evaluate_classifier.py (whichscores the frozen-encoder baseline). It fine-tunes a cross-encoder transformerover LectureBank pairs and reports precision/recall/F1 with the SAME nestedthreshold-selection methodology (threshold picked on a held-out val slice,never the test fold).Runs identically on:  * Local CPU:  & D:\\Anaconda3\\envs\\lecgap\\python.exe scripts/kaggle_fine_tune.py --epochs 1  * Kaggle GPU: python scripts/kaggle_fine_tune.py --input-dir /kaggle/input/datasets/ayushdevadiga/lecturebank \\                   --output-dir /kaggle/working --epochs 3CSVs expected (upload as a Kaggle dataset under your account — adjust theinput-dir if your dataset slug differs):  prerequisite_annotation.csv :: (Source_Topic_ID, Target_Topic_ID, If_prerequisite)  208topics.csv               :: (id, Topic, Topic_Link)Export: the model fine-tuned on ALL data is written to <output-dir>/model/ fordownload and CPU inference in the main pipeline."""import argparseimport csvimport jsonimport osimport sysfrom collections import Counterimport numpy as npfrom sklearn.metrics import f1_score, precision_score, recall_scorefrom sklearn.model_selection import StratifiedKFoldsys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))from experiments.fine_tune import (    build_train_triples,    export_model,    fine_tune_cross_encoder,    predict_pairs,)def load_data(annot_file, topics_file):    name_of = {}    with open(topics_file, newline="", encoding="utf-8") as f:        for row in csv.reader(f):            if len(row) >= 2:                name_of[row[0]] = row[1]    pairs = []    with open(annot_file, newline="", encoding="utf-8") as f:        for src, tgt, label in csv.reader(f):            if src in name_of and tgt in name_of:                pairs.append((name_of[src], name_of[tgt], int(label)))    return name_of, pairsdef run_cv(X, y, *, max_neg_ratio, epochs, batch_size, lr, base_model, weight_decay, grad_clip, device):    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)    metrics = []    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y)):        tr_idx = list(tr_idx)        vskf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)        fit_idx, val_idx = next(iter(vskf.split(tr_idx, [y[i] for i in tr_idx])))        fit_idx = [tr_idx[i] for i in fit_idx]        val_idx = [tr_idx[i] for i in val_idx]        tr_pairs = [X[i] for i in fit_idx]        tr_labels = [y[i] for i in fit_idx]        va_pairs = [X[i] for i in val_idx]        va_labels = [y[i] for i in val_idx]        te_pairs = [X[i] for i in te_idx]        te_labels = [y[i] for i in te_idx]        triples = build_train_triples(tr_pairs, tr_labels, max_neg_ratio=max_neg_ratio)        model, tok = fine_tune_cross_encoder(            triples,            epochs=epochs,            batch_size=batch_size,            lr=lr,            base_model=base_model,            weight_decay=weight_decay,            grad_clip=grad_clip,            device=device,        )        va_logits = predict_pairs(model, tok, va_pairs)        best = (0.0, 0.0)        for t in np.arange(-2.0, 3.0, 0.1):            preds = [1 if s >= t else 0 for s in va_logits]            f = f1_score(va_labels, preds, zero_division=0)            if f > best[0]:                best = (f, float(t))        thr = best[1]        te_logits = predict_pairs(model, tok, te_pairs)        preds = [1 if s >= thr else 0 for s in te_logits]        m = {            "p": precision_score(te_labels, preds, zero_division=0),            "r": recall_score(te_labels, preds, zero_division=0),            "f1": f1_score(te_labels, preds, zero_division=0),            "acc": sum(p == t for p, t in zip(preds, te_labels)) / len(te_labels),            "thr": thr,        }        metrics.append(m)        print(            f"  fold {fold+1}: P={m['p']:.3f} R={m['r']:.3f} "            f"F1={m['f1']:.3f} acc={m['acc']:.3f} @thr={thr:.2f}",            flush=True,        )    n = len(metrics)    avgs = {        "p": sum(m["p"] for m in metrics) / n,        "r": sum(m["r"] for m in metrics) / n,        "f1": sum(m["f1"] for m in metrics) / n,        "acc": sum(m["acc"] for m in metrics) / n,    }    print("\n=== 5-fold CV (fine-tuned) summary ===")    print(f"Precision (avg): {avgs['p']:.3f}")    print(f"Recall    (avg): {avgs['r']:.3f}")    print(f"F1        (avg): {avgs['f1']:.3f}")    print(f"Accuracy  (avg): {avgs['acc']:.3f}")    return {"avgs": avgs, "folds": metrics}def _train_val_test_split(X, y, random_state=42):    """Return (train_pairs, train_labels, val_pairs, val_labels, test_pairs, test_labels)    from a single stratified split — used for fast hyperparameter selection."""    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)    tr_idx, te_idx = next(iter(skf.split(X, y)))    tr_idx = list(tr_idx)    vskf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)    fit_idx, val_idx = next(iter(vskf.split(tr_idx, [y[i] for i in tr_idx])))    fit_idx = [tr_idx[i] for i in fit_idx]    val_idx = [tr_idx[i] for i in val_idx]    return (        [X[i] for i in fit_idx], [y[i] for i in fit_idx],        [X[i] for i in val_idx], [y[i] for i in val_idx],        [X[i] for i in te_idx], [y[i] for i in te_idx],    )def tune_hyperparams(X, y, *, epochs_grid, lr_grid, ratio_grid, batch_size, base_model, weight_decay, grad_clip, device):    """Single-split hyperparameter sweep over a small grid.    Returns the best (epochs, lr, max_neg_ratio) by val-F1. Fast on GPU —    each config trains once on the fit slice. The test slice is never touched    during selection, so the eventual 5-fold CV remains fair.    """    tr_pairs, tr_labels, va_pairs, va_labels, _te_pairs, _te_labels = _train_val_test_split(X, y)    results = []    for e in epochs_grid:        for lr in lr_grid:            for ratio in ratio_grid:                triples = build_train_triples(tr_pairs, tr_labels, max_neg_ratio=ratio)                model, tok = fine_tune_cross_encoder(                    triples, epochs=e, batch_size=batch_size, lr=lr,                    base_model=base_model, weight_decay=weight_decay,                    grad_clip=grad_clip, device=device,                )                va_logits = predict_pairs(model, tok, va_pairs)                best_f, best_t = (0.0, 0.0)                for t in np.arange(-2.0, 3.0, 0.1):                    preds = [1 if s >= t else 0 for s in va_logits]                    f = f1_score(va_labels, preds, zero_division=0)                    if f > best_f:                        best_f, best_t = f, float(t)                results.append((best_f, e, lr, ratio, best_t))                print(f"  tune e={e} lr={lr} ratio={ratio}: val-F1={best_f:.3f} @thr={best_t:.2f}",                      flush=True)    results.sort(key=lambda r: -r[0])    best_f, e, lr, ratio, t = results[0]    print("\n=== Best hyperparams (by val-F1) ===")    print(f"epochs={e}, lr={lr}, max_neg_ratio={ratio}  (val-F1 {best_f:.3f} @thr={t:.2f})")    return e, lr, ratiodef _resolve_input_dir(input_dir: str) -> str:    """Return a directory containing both lecturebank CSVs.    If ``input_dir`` already holds them, use it as-is. Otherwise, on Kaggle,    scan the standard input roots for the two CSVs (the dataset slug is not    always predictable — e.g. /kaggle/input/datasets/<user>/<slug>/).    """    def present(d):        return os.path.isfile(os.path.join(d, "prerequisite_annotation.csv")) and \            os.path.isfile(os.path.join(d, "208topics.csv"))    if present(input_dir):        return input_dir    roots = []    if os.path.isdir("/kaggle/input"):        roots.append("/kaggle/input")    for root in roots:        for dirpath, dirnames, filenames in os.walk(root):            if "prerequisite_annotation.csv" in filenames and "208topics.csv" in filenames:                print(f"Found lecturebank data at {dirpath}", flush=True)                return dirpath    return input_dirdef main():    ap = argparse.ArgumentParser()    ap.add_argument("--input-dir", default="data/lecturebank")    ap.add_argument("--output-dir", default="data/lecturebank/model")    ap.add_argument("--epochs", type=int, default=3)    ap.add_argument("--batch-size", type=int, default=32)    ap.add_argument("--lr", type=float, default=2e-5)    ap.add_argument("--max-neg-ratio", type=int, default=8)    ap.add_argument("--base-model", default=None,                    help="HF model id to fine-tune from (default fine_tune._DEF_BASE)")    ap.add_argument("--weight-decay", type=float, default=0.0,                    help="L2 weight decay on head params to fight overfitting")    ap.add_argument("--grad-clip", type=float, default=None,                    help="max gradient norm to clip (None = no clipping)")    ap.add_argument("--device", default=None)    ap.add_argument("--metrics-out", default=None,                    help="optional JSON path to dump the CV summary + chosen config")    ap.add_argument("--tune", action="store_true",                    help="run a fast single-split hyperparameter sweep first, "                         "then use the best config for the full CV + export")    ap.add_argument("--epochs-grid", default="2,3,4",                    help="comma-separated epochs to try during --tune")    ap.add_argument("--lr-grid", default="1e-5,2e-5,5e-5",                    help="comma-separated learning rates to try during --tune")    ap.add_argument("--ratio-grid", default="4,8,16",                    help="comma-separated max_neg_ratio values to try during --tune")    args = ap.parse_args()    input_dir = _resolve_input_dir(args.input_dir)    annot = os.path.join(input_dir, "prerequisite_annotation.csv")    topics = os.path.join(input_dir, "208topics.csv")    name_of, pairs = load_data(annot, topics)    X = [(a, b) for a, b, _ in pairs]    y = [l for _, _, l in pairs]    print(f"Loaded {len(pairs)} pairs across {len(name_of)} topics; "          f"balance {Counter(y)}", flush=True)    base_model = args.base_model    if base_model is None:        from experiments.fine_tune import _DEF_BASE        base_model = _DEF_BASE    print(f"Backbone: {base_model}", flush=True)    epochs, lr, max_neg_ratio = args.epochs, args.lr, args.max_neg_ratio    if args.tune:        eg = [int(s) for s in args.epochs_grid.split(",")]        lg = [float(s) for s in args.lr_grid.split(",")]        rg = [int(s) for s in args.ratio_grid.split(",")]        print("\nTuning hyperparameters on a single split...", flush=True)        epochs, lr, max_neg_ratio = tune_hyperparams(            X, y, epochs_grid=eg, lr_grid=lg, ratio_grid=rg,            batch_size=args.batch_size, base_model=base_model,            weight_decay=args.weight_decay, grad_clip=args.grad_clip,            device=args.device,        )    result = run_cv(        X, y,        max_neg_ratio=max_neg_ratio,        epochs=epochs,        batch_size=args.batch_size,        lr=lr,        base_model=base_model,        weight_decay=args.weight_decay,        grad_clip=args.grad_clip,        device=args.device,    )    # Final model on ALL data for deployment.    print("\nFine-tuning final model on all data...", flush=True)    triples = build_train_triples(X, y, max_neg_ratio=max_neg_ratio)    model, tok = fine_tune_cross_encoder(        triples, epochs=epochs, batch_size=args.batch_size,        lr=lr, base_model=base_model,        weight_decay=args.weight_decay, grad_clip=args.grad_clip,        device=args.device,    )    out = export_model(model, tok, args.output_dir)    print(f"Exported fine-tuned model to {out}", flush=True)    if args.metrics_out:        with open(args.metrics_out, "w", encoding="utf-8") as f:            json.dump({                "config": {                    "epochs": epochs, "lr": lr,                    "max_neg_ratio": max_neg_ratio, "batch_size": args.batch_size,                    "base_model": base_model, "weight_decay": args.weight_decay,                    "grad_clip": args.grad_clip, "used_tune": args.tune,                },                "cv": result,                "tuned_f1": round(result["avgs"]["f1"], 4),            }, f, indent=2)        print(f"Wrote metrics to {args.metrics_out}", flush=True)if __name__ == "__main__":    main()'''with open('/kaggle/working/kaggle_fine_tune.py', 'w') as f:    f.write(kaggle_src)print('wrote kaggle_fine_tune.py')

### 3. Train + evaluate + export (staging)MiniLM has been improving with every epoch bump (e3->0.49, e5->0.53, e8->0.554); this sweep pushes to 8/10/12 and adds weight-decay + grad-clip to close the val-vs-CV overfitting gap.Outputs go to `/kaggle/working/staging/` (model + `metrics.json`). Nothing is finalized yet — the **last cell** copies the model into the shipped `/kaggle/working/model/` folder only after you verify the CV F1 beats the frozen baseline (0.569).

In [ ]:
INPUT_DIR = '/kaggle/input/datasets/ayushdevadiga/lecturebank'STAGING = '/kaggle/working/staging'BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'# Each config trains ONCE on a single split; cost = (#configs) x (avg epochs)# epochs of T4 time. Backbone + grids are set from the notebook preset above.# weight-decay + grad-clip are in to curb the val-vs-CV overfitting gap.# The 'classifier' head + 'position_ids' LOAD-REPORT notes are suppressed# (expected task-shape noise, not errors).!python /kaggle/working/kaggle_fine_tune.py \    --input-dir {INPUT_DIR} \    --output-dir {STAGING}/model \    --metrics-out {STAGING}/metrics.json \    --base-model {BASE_MODEL} \    --batch-size 32 \    --weight-decay 0.01 --grad-clip 1.0 \    --tune \    --epochs-grid 8,10,12 \    --lr-grid 2e-5,5e-5 \    --ratio-grid 8print('\nStaging done. Model in {STAGING}/model, metrics in {STAGING}/metrics.json')print('Now run the LAST cell to ship the model only if staged F1 beats the baseline.')

### 4. FINALIZE — store the model outputs (run manually)**Run this cell only after checking the staged CV F1 (below) beats the frozen baseline 0.569.**It reads `staging/metrics.json`, prints the summary, and if `F1 > 0.569` copies the fine-tuned model into the shipped `/kaggle/working/model/` folder (what you download and drop into the repo). If the run is not better, **don't run this cell** — the final `model/` folder simply won't be produced.> Make sure you run this cell in the SAME kernel session as the training cell above (i.e. after cell 3 has finished), or `staging/metrics.json` won't exist yet.

In [ ]:
import json, os, shutil, globBASELINE_F1 = 0.569  # frozen-encoder benchmark (evaluate_classifier.py)STAGING = '/kaggle/working/staging'FINAL = '/kaggle/working/model'with open(os.path.join(STAGING, 'metrics.json')) as f:    m = json.load(f)cfg = m['config']cv = m['cv']['avgs']f1 = cv['f1']print('Tuned config : epochs=%s, lr=%s, max_neg_ratio=%s' % (    cfg['epochs'], cfg['lr'], cfg['max_neg_ratio']))print('CV F1        : %.3f  (P %.3f, R %.3f, acc %.3f)' % (    f1, cv['p'], cv['r'], cv['acc']))print('Baseline F1  : %.3f' % BASELINE_F1)beat = f1 > BASELINE_F1print('F1 beats baseline? %s' % beat)if not beat:    raise SystemExit('NOT finalized: tuned F1 does not beat the frozen '                     'baseline (0.569). No model/ folder was produced.')if os.path.isdir(FINAL):    shutil.rmtree(FINAL)shutil.copytree(os.path.join(STAGING, 'model'), FINAL)with open(os.path.join(FINAL, 'metrics.json'), 'w') as f:    json.dump(m, f, indent=2)print('Model stored in ' + FINAL + ':')for p in sorted(glob.glob(FINAL + '/**/*', recursive=True)):    if os.path.isfile(p):        print('  ', p, os.path.getsize(p))print('Download /kaggle/working/model/ and drop it into the repo as '      'data/models/lecgap_ft/ for CPU inference.')